# Exercise 1: Where does the air come from?

In this exercise, we are going to study where the air over a region of your choice comes from. For this purpose, we are going to run a _backward_ simulation with FLEXPART.

In the [COMMAND](https://flexpart.img.univie.ac.at/docs/configuration.html#command) file:
- Set the direction of the simulation to backward.
- Select a simulation length of 5 days. The start and end times should be within 2017-09-25 and 2017-10-03.
- Set the output time step to 1 hour.

In the [RELEASES](https://flexpart.img.univie.ac.at/docs/configuration.html#releases) file:
- Define two instantaneous releases toward the end date of the simulation (e.g. one day before the end and at the end).
- Set the species to "air tracer".
- Choose a reasonable number of particles to release. More particles give a more representative picture but increase the simulation time and memory use. 10000 is a good number for our purpose.
- Choose a release point or box in a region and height you are interested in. It should be within 0°E-90°E and 0°N-90°N, because this is the domain of the input data.
- The release mass is irrelevant for backward simulations.

In the [OUTGRID](https://flexpart.img.univie.ac.at/docs/configuration.html#outgrid) file:
- Select two output heights (e.g. 100 m and 10000 m above ground).
- Define the output grid centered on the release point/box. It should cover at least the area where you expect the air to come from, but should not be too big to save memory and plotting time.

In the [PARTOPTIONS](https://flexpart.img.univie.ac.at/docs/configuration.html#partoptions) file:
- Set variables to true that you would like to have in the particle output (e.g. altitude z, temperature T).

In the [AGECLASSES](https://flexpart.img.univie.ac.at/docs/configuration.html#ageclasses) file:
- Define two ageclasses of 2 days (= 172800 seconds) and 4 days (= 345600 seconds).

Submit the simulation. It will take a few minutes to finish. You can track the current simulation time in the runtime log.

In [ ]:
import numpy as np
import sys
sys.path.append("../util")
from IPython.display import HTML
from flexpart_io import *
from flexpart_plotting import *

In [ ]:
# Flexpart output directory
flxdir = "./run/output/"

## 1.1) Load gridded data

In [ ]:
ds_grid, _ = open_flexpart_grid(flxdir, fileformat="netcdf")
ds_grid

## 1.2) Define colormap and levels

In [ ]:
# colormap and levels
cmap_grid = plt.cm.BuPu

logmax = int(np.floor(np.log10(np.max(ds_grid["spec001_mr"]))))

levels = 10 ** np.arange(logmax - 5, logmax + 1e-10, 0.2)
norm = mcolors.LogNorm(vmin=levels[0], vmax=levels[-1])

## 1.3) Plot the plume at a specific time step on a map
To see where the air comes from, we can plot the plume at a specific time step. Select the time step you want to plot (e.g. 120 corresponds to 5 days if you have hourly output).

In [ ]:
# select indices
timestep = 83
pointspec = [0,1]
nageclass = [0,1]
height = [0]

# plot map
fig = plot_map_grid(ds_grid.isel(time=[timestep], pointspec=pointspec, nageclass=nageclass, height=height), cmap_grid, levels, norm)

**Notes on the result**: 
- The unit "s" refers to the average time particles spend in each grid cell at the given time step in seconds. The area-integrated value corresponds to FLEXPART's output time step.
- If you plot both releases, you see their plumes on top of each other.

## 1.4) Plot animation of the plume
We can also plot an animation over multiple time steps. You can adapt the timesteps below. If you want to plot all timesteps you can set `timesteps = slice(None)`. This may take some time.

In [ ]:
# select indices
timesteps = slice(0, 84)
pointspec = [0,1]
nageclass = [0,1]
height = [0]

# plot animation
ani = plot_map_grid_anim(ds_grid.isel(time=timesteps, pointspec=pointspec, nageclass=nageclass, height=height), cmap_grid, levels, norm)
HTML(ani.to_jshtml())

## 1.5) Load particle data

In [ ]:
ds_part = open_flexpart_part(flxdir)
partind = np.random.choice(ds_part["particle"], 10000) # select 10000 particles for faster plotting
ds_part_sel = ds_part.sel(particle=partind)
ds_part_sel

## 1.6) Define variable name, colormap, and levels

In [ ]:
varname = "z"
vmin = 0
vmax = 3000
cmap_part = plt.cm.cividis

## 1.7) Plot particles at a specific timestep on a map

In [ ]:
timestep = 83

# setup figure based on gridded file
fig, ax, gs = setup_fig(ds_grid)

# plot particles on the map
fig = plot_map_part(fig, ax, gs, ds_part_sel.isel(time=[timestep]), varname, cmap_part, vmin, vmax)

**Notes on the result**: The particle output does not care about the outgrid domain. This is only relevant for the gridded output.

## 1.8) Plot animation of particles

In [ ]:
timesteps = slice(0, 84)

# setup figure based on gridded file
fig, ax, gs = setup_fig(ds_grid)

# plot particles on the map
ani = plot_map_part_anim(fig, ax, gs, ds_part_sel.isel(time=timesteps), varname, cmap_part, vmin, vmax)
HTML(ani.to_jshtml())

## 1.9) Plot timeseries of particle properties
The particle output allows us to see the evolution of particle properties over time.

In [ ]:
fig = plot_timeseries_part(ds_part_sel, varname=varname, ylim=(vmin, vmax))

**Notes on the result**: 
- If you have multiple releases, you will see breaks in the timeseries at the time of the second release and another when the first release reaches its age as defined in the ageclasses.
- Do you see strange jumps in altitude in the lowest ~2000 m? This is Flexpart's boundary layer parameterization.

## Thoughts on exercise 1

Did the result surprise you or was it what you expected? Do you know how it compares to the climatology?

Climatological transport patterns could be generated for example with the Lagrangian Reanalysis (LARA) dataset, which is available here: https://data.eodc.wolke.img.univie.ac.at/

Instead of looking at the air tracer, we could do the same for other gases or aerosols. This will be covered in the upcoming sessions.